# RAG 실습 노트북

PDF 한 개를 챗봇으로 만드는 과정을 9단계로 따라간다. 이 노트북은 step1
부터 step6 까지 (코어 RAG 흐름) 를 셀 단위로 실행해보는 용. step7~9 는
Streamlit 앱이라 별도 터미널에서 `streamlit run` 으로 돌린다.

사전에 `.env` 파일에 `BACKEND` 와 키를 채워두자. 자세한 셋업은
[README.md](README.md), 명령어만 보고 싶으면 [COMMANDS.md](COMMANDS.md).


## 사전: 환경 변수 + 백엔드 확인

`.env` 가 제대로 로드됐는지, 어떤 백엔드를 쓰는지 한 번 찍어본다.


In [ ]:
from models import describe_backend
print(describe_backend())


## Step 1. PDF 로드

PyPDFLoader 는 페이지 한 장을 LangChain 의 `Document` 한 개로 만든다.


In [ ]:
from langchain_community.document_loaders import PyPDFLoader

pages = PyPDFLoader("출장 규정.pdf").load()
print(f"총 {len(pages)}페이지 로드.")


In [ ]:
# 첫 페이지 앞부분 200자
print(pages[0].page_content[:200])


In [ ]:
# 메타데이터에 페이지 번호와 원본 파일 정보가 같이 들어있다
pages[0].metadata


## Step 2. 청크 분할

페이지 통으로 LLM 에 던지기엔 길고, 너무 잘게 자르면 문맥이 끊긴다.
`RecursiveCharacterTextSplitter` 가 적당한 크기 + 약간의 overlap 으로
잘라준다.

두 시나리오를 비교해보자.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

small = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
large = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)

chunks_small = small.split_documents(pages)
chunks_large = large.split_documents(pages)

print(f"300자 기준  -> {len(chunks_small)}개 청크")
print(f"1000자 기준 -> {len(chunks_large)}개 청크")


In [ ]:
# 줄글 보고서/규정엔 1000자가 자연스럽다
chunks = chunks_large
print(f"이후엔 {len(chunks)}개 청크로 진행")


In [ ]:
# 첫 청크 내용
print(chunks[0].page_content[:300])
print()
print("메타데이터:", chunks[0].metadata)


## Step 3. 임베딩 + FAISS 저장

청크를 벡터로 바꿔서 FAISS 인덱스에 넣는다. OpenAI 백엔드면 API 호출,
HF 백엔드면 로컬 sentence-transformers (첫 실행 시 가중치 다운로드).

한 번 만든 인덱스는 디스크에 저장해서 다음 단계들이 재사용한다.


In [ ]:
from langchain_community.vectorstores import FAISS
from models import get_embeddings

embeddings = get_embeddings()
print("임베딩 중...")
vector_db = FAISS.from_documents(chunks, embeddings)
vector_db.save_local("my_faiss_index")

print(f"저장된 벡터 수: {vector_db.index.ntotal}")
print(f"차원: {vector_db.index.d}")


## Step 4. 유사도 검색

질문을 같은 임베더로 벡터화해서 가까운 청크 k 개를 찾는다.
LLM 호출 없이 "검색만" 따로 확인하는 단계.


In [ ]:
query = "해외 출장 갔을 때 돈 어떻게 받아?"
docs = vector_db.similarity_search(query, k=3)

for i, doc in enumerate(docs, start=1):
    page = doc.metadata.get("page", "?")
    print(f"[{i}위] p.{page}")
    print(f"    {doc.page_content[:200]}...")
    print()


In [ ]:
# 점수(거리)와 함께 — 작을수록 더 유사
for i, (doc, score) in enumerate(
    vector_db.similarity_search_with_score(query, k=3), start=1
):
    print(f"{i}. distance={score:.4f}  p.{doc.metadata.get('page','?')}")


## Step 5. LLM 답변 생성

검색해 온 청크들을 프롬프트의 context 자리에 끼워 넣고 LLM 에 던진다.
이게 RAG 의 G(Generation) 부분.


In [ ]:
from langchain_core.prompts import PromptTemplate
from models import get_llm

# 검색 결과 3개를 합쳐서 context 로 만든다
context = "\n\n---\n\n".join(
    f"(p.{d.metadata.get('page','?')})\n{d.page_content}"
    for d in docs
)

prompt = PromptTemplate.from_template(
    """당신은 사내 규정 안내 챗봇입니다. 아래 [검색된 문서] 만 참고하여 답변하세요.

[검색된 문서]
{context}

[사용자 질문]
{question}

[답변]"""
)

llm = get_llm(temperature=0)
chain = prompt | llm


In [ ]:
response = chain.invoke({"context": context, "question": query})
print(response.content)


## Step 6. 프롬프트로 답변 통제

같은 검색 결과라도 프롬프트가 어떻게 쓰여있느냐에 따라 답이 크게 달라진다.

- 문서에 없는 질문에도 LLM 이 자기 지식으로 환각하는 걸 막고 싶다 →
  "문서에 없으면 답하지 마라" 라고 박는다.
- 형식(불릿, 길이)을 통일하고 싶다 → 그것도 프롬프트에서 강제.


In [ ]:
loose_prompt = PromptTemplate.from_template(
    """당신은 사내 규정 안내 챗봇입니다. 아래 [검색된 문서] 를 참고하여 답변하세요.

[검색된 문서]
{context}

[사용자 질문]
{question}"""
)

strict_prompt = PromptTemplate.from_template(
    """당신은 사내 규정 안내 챗봇입니다. 반드시 아래 [검색된 문서] 만 참고하세요.
문서에 정답이 없다면 절대 지어내지 말고 정확히 "문서에서 찾을 수 없습니다" 라고만 답하세요.
답변은 불릿 포인트(•)로 3줄 이내로 요약하세요.

[검색된 문서]
{context}

[사용자 질문]
{question}"""
)


def ask(prompt_template, question):
    found = vector_db.similarity_search(question, k=3)
    ctx = "\n\n---\n\n".join(d.page_content for d in found)
    chain = prompt_template | llm
    return chain.invoke({"context": ctx, "question": question}).content


### 비교 1) 문서에 답이 있는 질문


In [ ]:
q1 = "해외 출장 갔을 때 돈 어떻게 받아?"

print("[느슨한 프롬프트]")
print(ask(loose_prompt, q1))
print()
print("[엄격한 프롬프트]")
print(ask(strict_prompt, q1))


### 비교 2) 문서에 없는 질문 — 환각 테스트


In [ ]:
q2 = "우리 회사 대표이사 이름이 뭐야?"

print("[느슨한 프롬프트]")
print(ask(loose_prompt, q2))
print()
print("[엄격한 프롬프트]")
print(ask(strict_prompt, q2))


## Step 7~9. Streamlit 챗봇

여기까진 콘솔/노트북에서 돌렸다. 사용자 입장의 인터페이스를 챗 형태로
만들려면 Streamlit 이 필요한데, 이건 노트북 안에서 실행되지 않고 별도
프로세스로 띄운다.

새 터미널에서:
```bash
source .venv/bin/activate
streamlit run step7_chat_ui.py        # 채팅 UI 골격
streamlit run step8_file_upload.py    # 파일 업로드 위젯
streamlit run step9_app_complete.py   # 전부 합친 완성판
```

step9 가 최종 챗봇이다. PDF 업로드 → 자동 인덱싱 → 질문 → 답변 + 출처
표시까지 한 화면에서 동작한다.

각 파일 안 코드를 직접 열어 보면 step1~6 의 흐름이 그대로 들어있는 게
보인다 — 차이는 입력/출력이 콘솔이 아니라 Streamlit 위젯이라는 것뿐.


## 마무리

이 9단계가 RAG 시스템의 최소 골격이다. 상위 폴더의 메인 앱(`../app.py`)
은 같은 흐름 위에 다음이 얹혀 있다.

- 멀티프로바이더 (OpenAI / Anthropic / Fireworks / HF Router / DashScope / vLLM)
- 하이브리드 검색 (BM25 + Dense + RRF + Cross-encoder rerank)
- HyDE / Multi-query / Contextual rewrite
- 멀티모달 PDF 페이지 이미지
- Supabase 영속 저장 + 사용자 인증
- 에이전트 워크플로 (이메일 / 보고서 / 요약 / 분석 / 비교)

기본 동작 원리를 여기서 익혀두면 메인 앱 코드를 읽기가 한결 편하다.
